[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ODJ0930/ComplexSync/blob/main/ComplexSync_Colab.ipynb)

# ComplexSync on Colab

Lip-syncs a video to a new audio track with [ComplexSync](https://github.com/Playmate111/ComplexSync) (inference only).

1. *Runtime → Change runtime type* → pick a GPU (H100 by default).
2. Put your input video and audio in Google Drive under `MyDrive/complexsync/inputs/`, or upload them when step 3 asks.
3. Run all cells. The result is saved to `MyDrive/complexsync/outputs/`.

Model weights are downloaded fresh each session (about 5.5 GB per stage from ModelScope).

### 1. Install

In [ ]:
!git clone --depth 1 https://github.com/ODJ0930/ComplexSync.git /content/ComplexSync
%cd /content/ComplexSync
!pip install -q -r requirements.txt modelscope
# InsightFace needs the CUDA build of onnxruntime; drop the CPU build if something pulled it in.
!pip uninstall -y -q onnxruntime
!pip install -q --force-reinstall --no-deps onnxruntime-gpu

### 2. Download models

In [ ]:
MODEL_TYPE = "stage3"  # @param ["stage3", "stage1"]
# stage3: 1 denoising step, about 10x faster, also restores occlusions.
# stage1: 12 steps with classifier-free guidance.

import os, urllib.request
from huggingface_hub import snapshot_download

os.chdir("/content/ComplexSync")
os.makedirs("models/whisper", exist_ok=True)

# ComplexSync checkpoint (ModelScope)
if not os.path.exists(f"models/{MODEL_TYPE}.pt"):
    !modelscope download --model alkdjfklaj/ComplexSync {MODEL_TYPE}.pt --local_dir models

# Stable Diffusion VAE
snapshot_download(
    "stabilityai/sd-vae-ft-mse",
    local_dir="models/sd-vae-ft-mse",
    allow_patterns=["config.json", "diffusion_pytorch_model.safetensors"],
)

# Whisper tiny encoder (both released configs use cross_attention_dim 384)
WHISPER_TINY = "https://openaipublic.azureedge.net/main/whisper/models/65147644a518d12f04e32d6f3b26facc3f8dd46e5390956a9424a650c0ce22b9/tiny.pt"
if not os.path.exists("models/whisper/tiny.pt"):
    urllib.request.urlretrieve(WHISPER_TINY, "models/whisper/tiny.pt")

!ls -la models models/whisper

### 3. Inputs from Google Drive

In [ ]:
VIDEO_NAME = "demo.mp4"  # @param {type:"string"}
AUDIO_NAME = "demo.wav"  # @param {type:"string"}

import shutil, subprocess
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive")
WORK = Path("/content/drive/MyDrive/complexsync")
IN_DIR, OUT_DIR = WORK / "inputs", WORK / "outputs"
IN_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

for name in (VIDEO_NAME, AUDIO_NAME):
    if not (IN_DIR / name).exists():
        print(f"{name} is not in {IN_DIR}; upload it now:")
        for uploaded, data in files.upload().items():
            (IN_DIR / uploaded).write_bytes(data)

# Work on local copies (the inference code shells out to ffmpeg with unquoted paths).
LOCAL = Path("/content/inputs")
LOCAL.mkdir(exist_ok=True)
video_in = LOCAL / "video_25fps.mp4"
audio_in = LOCAL / ("audio" + Path(AUDIO_NAME).suffix)
# The model works at 25 FPS, so resample the video first.
subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", str(IN_DIR / VIDEO_NAME),
                "-r", "25", "-crf", "18", str(video_in)], check=True)
shutil.copy(IN_DIR / AUDIO_NAME, audio_in)
print("video:", video_in, "\naudio:", audio_in)

### 4. Run lip sync

In [ ]:
import time
from IPython.display import Video, display

os.chdir("/content/ComplexSync")
out_local = Path("/content/outputs") / f"{Path(VIDEO_NAME).stem}_{MODEL_TYPE}.mp4"
out_local.parent.mkdir(exist_ok=True)

!python -m scripts.inference \
    --model_type {MODEL_TYPE} \
    --inference_ckpt_path models/{MODEL_TYPE}.pt \
    --video_path {video_in} \
    --audio_path {audio_in} \
    --video_out_path {out_local}

if out_local.exists():
    saved = OUT_DIR / f"{out_local.stem}_{time.strftime('%Y%m%d-%H%M%S')}.mp4"
    shutil.copy(out_local, saved)
    print("Saved to Drive:", saved)
    display(Video(str(out_local), embed=True, width=640))
else:
    print("No output produced; check the log above.")